In [1]:
import sys

sys.path.append('c:/users/jijing/appdata/roaming/pypoetry/venv/lib/site-packages')

In [2]:
import pandas as pd
import numpy as np

data = pd.read_csv('HMEQPERF_all_pred.csv')

col_types: dict[str, str] = {
    'DELINQ': 'int',
    'DEBTINC': 'float',
    'VALUE': 'float',
    'DEROG': 'int',
    'CLNO': 'int',
    'CLAGE': 'float',
    'LOAN': 'float',
    'MORTDUE': 'float',
    'REASON': 'str',
    'JOB': 'str',
    'YOJ': 'float',
    'NINQ': 'int'
}

data.replace('           .', np.nan, inplace=True)
for col in data.columns:
    if col in col_types:
        if col_types[col] == 'int' or col_types[col] == 'float':
          data[col] = data[col].astype(float)

data

,DELINQ,DEBTINC,VALUE,DEROG,CLNO,CLAGE,LOAN,MORTDUE,REASON,JOB,YOJ,NINQ,BAD,EM_CLASSIFICATION
0,0.0,NaN,27249.538380,0.0,42.0,190.800000,18000.000000,42112.048060,DebtCon,ProfExe,10.000000,1.0,0,0
1,0.0,NaN,2731.915722,0.0,16.0,108.533333,10372.078010,48079.990390,DebtCon,NaN,4.000000,0.0,0,1
2,0.0,NaN,160800.000000,0.0,11.0,129.833333,18000.000000,62000.000000,HomeImp,ProfExe,15.000000,1.0,1,1
3,0.0,NaN,21500.000000,3.0,33.0,109.566667,5382.021035,12900.000000,HomeImp,Office,5.000000,0.0,1,1
4,0.0,NaN,2206.248555,0.0,14.0,165.333333,18000.000000,57988.000000,DebtCon,Other,12.334238,2.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23835,0.0,36.262691,27331.121600,0.0,15.0,193.702051,66652.458470,3707.072795,DebtCon,Other,16.000000,0.0,0,0
23836,0.0,34.751158,36614.466540,0.0,16.0,214.426206,53890.925670,50240.000000,DebtCon,Other,6.949012,0.0,0,0
23837,0.0,34.242465,94058.000000,0.0,15.0,218.304978,25775.741990,53307.000000,DebtCon,Other,16.000000,0.0,0,0
23838,1.0,34.818262,40379.532450,0.0,15.0,205.650160,38319.775950,2347.704447,DebtCon,Other,12.544639,0.0,0,0


In [3]:
import random
from typing import Any


INJECT_COUNT = 50
INJECT_DIM = 6

ignore_cols = []
target_col = 'BAD'
pred_col = 'EM_CLASSIFICATION'

candidate_cols = [c for c in data.columns]
for col in ignore_cols:
  candidate_cols.remove(col)
candidate_cols.remove(target_col)
candidate_cols.remove(pred_col)


injections = []
for i in range(INJECT_COUNT):
  inject_cols = np.random.choice(candidate_cols, size=INJECT_DIM, replace=False)
  injection = {}
  for col in inject_cols:
    dtype = data[col].dtype
    val: Any
    if col_types[col] == 'float':
      max_val = data[col].max()
      min_val = data[col].min()
      while True:
        val = random.uniform(min_val, max_val)
        val = round(val, 2)
        if val not in data[col].unique():
          break
    elif col_types[col] == 'str' or col_types[col] == 'int':
      val = "mock" + str(i)
    else:
      raise Exception('Unknown type of column ' + str(dtype))
    injection[str(col)] = val
  injections.append(injection)

injections

[{'CLNO': 'mock0',
  'JOB': 'mock0',
  'NINQ': 'mock0',
  'VALUE': np.float64(87644.72),
  'LOAN': np.float64(65776.08),
  'YOJ': np.float64(6.98)},
 {'DEROG': 'mock1',
  'MORTDUE': np.float64(4476.11),
  'NINQ': 'mock1',
  'DEBTINC': np.float64(99.32),
  'JOB': 'mock1',
  'LOAN': np.float64(11296.98)},
 {'NINQ': 'mock2',
  'REASON': 'mock2',
  'YOJ': np.float64(36.53),
  'CLAGE': np.float64(989.44),
  'DELINQ': 'mock2',
  'DEBTINC': np.float64(88.69)},
 {'CLAGE': np.float64(978.67),
  'DELINQ': 'mock3',
  'DEBTINC': np.float64(119.76),
  'VALUE': np.float64(370347.06),
  'REASON': 'mock3',
  'YOJ': np.float64(23.6)},
 {'YOJ': np.float64(40.25),
  'JOB': 'mock4',
  'VALUE': np.float64(133427.92),
  'DEROG': 'mock4',
  'REASON': 'mock4',
  'DEBTINC': np.float64(135.49)},
 {'CLAGE': np.float64(789.85),
  'VALUE': np.float64(501637.13),
  'CLNO': 'mock5',
  'DEBTINC': np.float64(21.26),
  'REASON': 'mock5',
  'MORTDUE': np.float64(315136.65)},
 {'CLNO': 'mock6',
  'DEROG': 'mock6',
  'YOJ

In [4]:
import math
from typing import Any
import numpy as np

base_err_count: int = np.count_nonzero(data[target_col] != data[pred_col]) 
base_err_rate = base_err_count / len(data)

inject_err_rate_inc: float = 0.6
inject_err_rate = base_err_rate + inject_err_rate_inc
inject_err_cov = 0.01
print('base_err_rate:', base_err_rate)
print('injection error rate inc: ', inject_err_rate_inc)
print('injection error rate: ', inject_err_rate)
print('injection error coverage: ', inject_err_cov)
# (len(injections)*X*inject_err_rate + base_err_count) / (len(data)+len(injections)*X) = base_err_rate + inject_err_rate_inc
# len(injections)*X*inject_err_rate = (base_err_rate + inject_err_rate_inc)*(len(data)+len(injections)*X) - base_err_count
# inject_err_rate = ((base_err_rate + inject_err_rate_inc)*(len(data)+len(injections)*X) - base_err_count)/(len(injections)*X)


# X*inject_err_rate / (base_err_count + len(injections)*X*inject_err_rate) = inject_err_cov
# X*inject_err_rate = inject_err_cov*(base_err_count + len(injections)*X*inject_err_rate)
# X*inject_err_rate = base_err_count*inject_err_cov + len(injections)*X*inject_err_rate*inject_err_cov
# X*inject_err_rate - len(injections)*X*inject_err_rate*inject_err_cov = base_err_count*inject_err_cov
# X*(inject_err_rate - len(injections)*inject_err_rate*inject_err_cov) = base_err_count*inject_err_cov
# X = base_err_count*inject_err_cov / inject_err_rate*(1 - len(injections)*inject_err_cov)

unique_val_map: dict[str, (list[Any], list[float])] = {}
for col in data.columns:
    if col in ignore_cols or col == target_col or col == pred_col:
        continue
    unique_val_map[col] = ([], [])
    val_prob = data[col].value_counts(normalize=True, dropna=False)
    for val, prob in val_prob.items():
        unique_val_map[col][0].append(val)
        unique_val_map[col][1].append(prob)

inject_count: int = math.ceil(base_err_count*inject_err_cov / (inject_err_rate - len(injections)*inject_err_rate*inject_err_cov))
print("count for each injection:", inject_count)
rows = []
pad_count = 10
min_actual_inject_err_rate: float = 1
for i in range(0, len(injections)):
    injection = injections[i]
    actual_error: int = 0
    for mock_col in injection:
        for j in range(0, pad_count):
            row = {target_col: 1}
            for col in data.columns:
                if col in ignore_cols:
                    row[col] = '?'
                elif col == target_col:
                    continue
                elif col == pred_col:
                    row[pred_col] = row[target_col]
                elif col == mock_col:
                    row[col] = injection[mock_col]
                else:
                    val = np.random.choice(unique_val_map[col][0], p=unique_val_map[col][1])
                    # val = np.random.choice(unique_val_map[col][0])
                    row[col] = val
            rows.append(row)
    for j in range(0, inject_count):
        row = {target_col: 1}
        for col in data.columns:
            if col in ignore_cols:
                row[col] = '-'
            elif col == target_col:
                continue
            elif col == pred_col:
                r = np.random.rand(1)[0]
                if r <= inject_err_rate:
                    row[pred_col] = int(not row[target_col])
                    actual_error += 1
                else:
                    row[pred_col] = row[target_col]
            elif col in injection:
                row[col] = injection[col]
            else:
                val = np.random.choice(unique_val_map[col][0], p=unique_val_map[col][1])
                # val = np.random.choice(unique_val_map[col][0])
                row[col] = val
        rows.append(row)
    actual_err_rate: float = actual_error / (pad_count + inject_count)
    if actual_err_rate < min_actual_inject_err_rate:
        min_actual_inject_err_rate = actual_err_rate

all_col_data = {}
for col in data.columns:
    col_data = []
    for row in rows:
        col_data.append(row[col])
    all_col_data[col] = col_data
append_data: pd.DataFrame = pd.DataFrame(all_col_data)

mock_data = pd.concat([data, append_data])
mock_data.to_csv('HMEQPERF_all_pred_mock.csv', index=False)
print('min_actual_inject_err_rate: %.2f' % min_actual_inject_err_rate)

    

base_err_rate: 0.17885906040268457
injection error rate inc:  0.6
injection error rate:  0.7788590604026846
injection error coverage:  0.01
count for each injection: 110
min_actual_inject_err_rate: 0.62


In [ ]:
# mdca analysis...
! mdca -d 'ctr_prediction_mock.csv' -m error -ic 'session_id,DateTime,user_id' -tc is_click -pc pred -mec 0.009 -mr=100 -nb -o 'testout.json'

In [40]:
import json

with open('testout.json', "r") as json_file:
    content = json.load(json_file)

res_str_set = set()
for i in range(len(content)):
    if content[i]['target_rate'] < min_actual_inject_err_rate:
        continue
    res_list = content[i]['items']
    res_dict = {}
    for item in res_list:
        val = item['value']
        if val == 'NaN':
            val = np.nan
        res_dict[item['column']] = val
    res_str = '['
    for col in data.columns:
        if col in res_dict:
            res_str += col + '=' + str(res_dict[col]) + ', '
    res_str = res_str[:-2]
    res_str += ']'
    res_str_set.add(res_str)


In [41]:
injection_str_set: set[str] = set()
for injection in injections:
    injection_str = '['
    for col in data.columns:
        if col in injection:
            injection_str += (col+'='+str(injection[col])+', ')
    injection_str = injection_str[:-2]
    injection_str += ']'
    injection_str_set.add(injection_str)

found: int = 0
for injection_str in injection_str_set:
    if injection_str in res_str_set:
        print('found:', injection_str)
        found += 1
    else:
        print('NOT found:', injection_str)

TP: int = found
FN: int = len(injections) - TP
TN: int = 0
FP: int = 0
for res_str in res_str_set:
    if res_str not in injection_str_set:
        print('NOT exist: ', res_str)
        FP += 1
print("TP: %d, FP: %d, FN: %d" % (TP, FP, FN))
recall: float = TP / (TP + FN)
precision: float = TP / (TP + FP)
accurate: float = (TP + TN) / (TP + TN + FP + FN)
f1: float = 2 * (precision*recall) / (precision+recall)
print('recall: %.2f%%' % (recall*100))
print('precision: %.2f%%' % (precision*100))
print('accurate: %.2f%%' % (accurate*100))
print('f1: %.2f%%' % (f1*100))


NOT found: [product=mock28, webpage_id=51481.61, product_category_2=301398.44, age_level=0.14, user_depth=2.05, var_1=0.61]
NOT found: [product=mock20, campaign_id=253644.75, webpage_id=55788.19, gender=mock20, age_level=2.23, user_depth=1.01]
NOT found: [product=mock33, product_category_2=423697.58, gender=mock33, age_level=5.83, user_depth=1.03, var_1=0.19]
NOT found: [city=mock35, product=mock35, campaign_id=384346.16, product_category_1=2.92, gender=mock35, age_level=0.41]
NOT found: [campaign_id=289610.42, webpage_id=41881.61, user_group_id=5.55, gender=mock40, age_level=2.51, var_1=0.33]
NOT found: [city=mock7, product=mock7, webpage_id=7871.17, product_category_1=3.64, user_group_id=1.28, age_level=0.93]
NOT found: [city=mock11, product=mock11, webpage_id=34647.71, gender=mock11, age_level=3.36, var_1=0.25]
NOT found: [city=mock41, product=mock41, webpage_id=8528.65, product_category_1=4.12, product_category_2=315237.13, gender=mock41]
NOT found: [product=mock42, webpage_id=5021

ZeroDivisionError: float division by zero

In [16]:
import pandas as pd
pd.set_option('display.max_rows', None)

d = pd.read_csv('HMEQPERF_all_pred.csv')
d['LOAN'].value_counts().sort_index()

LOAN
1.957958          1
2.126586          1
2.588011          1
2.717412          1
2.979822          1
10.211817         1
13.108949         1
16.082687         1
18.200302         1
18.803344         1
19.544235         1
26.887050         1
27.090299         1
27.580794         1
30.557139         1
32.223009         1
32.739791         1
34.358870         1
35.836671         1
36.205927         1
36.311903         1
41.193217         1
43.298929         1
43.685268         1
45.646538         1
45.805413         1
48.095684         1
51.276824         1
53.442732         1
54.379727         1
57.131561         1
58.204057         1
58.472152         1
59.191430         1
60.284468         1
62.060794         1
67.458063         1
72.317864         1
75.774907         1
77.317324         1
78.852020         1
80.174293         1
81.997696         1
83.774508         1
83.939247         1
84.576808         1
86.130984         1
86.430341         1
88.492203         1
91.430444      